In [0]:
from pyspark.sql.functions import col, explode

print("Iniciando a Camada Prata: Desaninhamento Estrutural (Citações)...")

# Ingestão da Camada Bronze (fonte de dados imutável)
df_bronze = spark.table("bronze_openalex")

# Desaninhamento do array de citações (flattening estrutural sem aplicar regras analíticas da Ouro)
df_citacoes_achatado = df_bronze.select(
    col("id").alias("work_id_origem"),
    explode(col("referenced_works")).alias("work_id_citado")
)

# Purificação estrutural: remoção de conexões nulas para garantir a integridade dos relacionamentos
df_citacoes_limpo = df_citacoes_achatado.filter(col("work_id_citado").isNotNull())

total_citacoes = df_citacoes_limpo.count()
print(f"Total de registros desaninhados e limpos na Prata (Citações): {total_citacoes}")

# Persistência física em formato colunar otimizado (Delta Lake)
tabela_prata_citacoes = "silver_citations_flattened"
df_citacoes_limpo.write.format("delta").mode("overwrite").saveAsTable(tabela_prata_citacoes)

print(f"\nTabela Delta '{tabela_prata_citacoes}' gerada com sucesso! O dado estrutural está pronto para a Camada Ouro.")
display(df_citacoes_limpo.limit(10))